# 06 | SQL Business Analysis

**Project:** Dubai Short-Term Rental Intelligence  
**Data source:** AirROI Dubai short-term rental dataset (third-party Kaggle release)  
**Coverage:** March 2021–February 2026  
**Analytical layer:** SQLite database constructed in `05_sql_database_setup.ipynb`

## Executive objective

Use advanced SQL to examine (1) the reporting sample and its historical coverage, (2) reported nightly-rate patterns, (3) comparable-property pricing deviations, and (4) the reliability of the resulting business signals.

**Stakeholders:** Short-term rental property managers, revenue-management analysts, and BI teams. This is a **historical sample analysis**, not a live Dubai market monitor. AirROI occupancy and revenue values are **provider estimates** rather than verified bookings or audited financial results; unusually priced properties are candidates for **manual review**, not proven mispricing.

## 1. Environment and analytical source

The notebook reads the validated SQLite database **without rebuilding or modifying it**. Run Notebook 05 first to generate `data/processed/dubai_rentals.sqlite`. The project-root detection supports launching Jupyter from the repository root or `notebooks/`. Only derived aggregates and figures are exported; original records remain outside the GitHub package pending license review.

**Techniques demonstrated:** common table expressions (CTEs), `LAG()`, `ROW_NUMBER()`, window `PARTITION BY`, conditional aggregation, `CASE`, date-aware cohort matching, ranking, and explicit reconciliation tests.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (current, *current.parents)
     if (folder / "README.md").is_file()
     and (folder / "data" / "processed" / "dubai_rentals.sqlite").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project database not found. Complete Notebook 05 first and launch Jupyter "
        "from the project root or notebooks/ directory."
    )
DB_PATH = PROJECT_ROOT / "data" / "processed" / "dubai_rentals.sqlite"
FIGURES = PROJECT_ROOT / "reports" / "figures"
REPORT_SQL = PROJECT_ROOT / "reports" / "sql"
FIGURES.mkdir(parents=True, exist_ok=True)
REPORT_SQL.mkdir(parents=True, exist_ok=True)

# Read-only access protects the database constructed in Notebook 05.
conn = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)
conn.execute("PRAGMA foreign_keys = ON")
conn.execute("PRAGMA query_only = ON")
assert conn.execute("PRAGMA foreign_keys").fetchone()[0] == 1
pd.set_option("display.max_columns", 24)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

def sql_frame(query: str, params=()) -> pd.DataFrame:
    return pd.read_sql_query(query, conn, params=params)

print("Source: AirROI Dubai | validated SQLite analytical layer")
print("SQLite:", sqlite3.sqlite_version)
print("Database mode: read-only")

## 2. Database readiness and record reconciliation

Validate the existence of required tables and views and check that the fact grain remains **one row per listing per month**. This establishes which observations form the denominator of each analysis. A database `integrity_check` alone does not validate business logic, so cross-table reconciliation is performed separately.

In [2]:
required_tables = {
    "dim_listing", "dim_month", "fact_listing_month", "fact_peer_benchmark",
    "fact_coverage_month", "peer_examples_latest",
}
required_views = {
    "vw_latest_price_review_queue", "vw_monthly_coverage_and_performance",
    "vw_pairwise_same_listing_month", "vw_latest_benchmark_status",
    "vw_segment_monthly_rates",
}
existing_tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
existing_views = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='view'")}
assert required_tables <= existing_tables, f"Missing tables: {required_tables-existing_tables}"
assert required_views <= existing_views, f"Missing views: {required_views-existing_views}"
assert conn.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
assert conn.execute("PRAGMA foreign_key_check").fetchall() == []

record_checks = sql_frame("""
WITH monthly AS (
    SELECT COUNT(*) AS observations,
           COUNT(DISTINCT listing_id) AS unique_listings,
           COUNT(DISTINCT month_date) AS reporting_months,
           MIN(month_date) AS first_month,
           MAX(month_date) AS last_month
    FROM fact_listing_month
)
SELECT *,
 (SELECT COUNT(*) FROM fact_peer_benchmark) AS benchmark_rows,
 (SELECT COUNT(*) FROM dim_listing) AS listing_dimension_rows,
 (SELECT COUNT(*) FROM dim_month) AS month_dimension_rows,
 (SELECT COUNT(*) FROM fact_coverage_month) AS coverage_rows,
 (SELECT COUNT(*) FROM (
     SELECT listing_id, month_date FROM fact_listing_month GROUP BY 1,2 HAVING COUNT(*) > 1
 )) AS duplicate_listing_month_keys
FROM monthly
""")
record = record_checks.iloc[0]
assert int(record.observations) == int(record.benchmark_rows)
assert int(record.unique_listings) == int(record.listing_dimension_rows)
assert int(record.reporting_months) == int(record.month_dimension_rows) == int(record.coverage_rows)
assert int(record.duplicate_listing_month_keys) == 0
coverage_key_issues = sql_frame("""
SELECT COUNT(*) AS mismatches FROM (
    SELECT c.month_date, c.observed_listings, c.active_listings,
           COUNT(f.listing_id) AS recomputed_observed,
           SUM(COALESCE(f.active,0)) AS recomputed_active
    FROM fact_coverage_month c
    LEFT JOIN fact_listing_month f ON f.month_date = c.month_date
    GROUP BY c.month_date
    HAVING c.observed_listings != COUNT(f.listing_id)
        OR c.active_listings != SUM(COALESCE(f.active,0))
)
""").iloc[0,0]
assert int(coverage_key_issues) == 0
latest_month = str(record.last_month)
display(record_checks)
print("Monthly coverage reconciliation: PASS (0 mismatched months)")
print("Latest reporting month:", latest_month)

## 3. Latest-month executive KPIs

Use **conditional aggregation** to explicitly distinguish observed listings from active listings. A median nightly rate is computed using SQL window rankings; it is not inferred from the mean. Monetary fields are AED. Estimated occupancy percentages use `occupancy × 100`.

In [3]:
latest_kpis = sql_frame("""
WITH active_rates AS (
    SELECT native_avg_daily_rate AS rate,
           ROW_NUMBER() OVER (ORDER BY native_avg_daily_rate) AS rn,
           COUNT(*) OVER () AS n
    FROM fact_listing_month
    WHERE month_date = ? AND active = 1 AND native_avg_daily_rate IS NOT NULL
),
median_price AS (
    SELECT AVG(rate) AS median_active_rate_aed
    FROM active_rates
    WHERE rn IN ((n + 1) / 2, (n + 2) / 2)
)
SELECT f.month_date,
       COUNT(*) AS observed_listings,
       SUM(CASE WHEN f.active=1 THEN 1 ELSE 0 END) AS active_listings,
       ROUND(AVG(CASE WHEN f.active=1 THEN f.native_avg_daily_rate END),2) AS mean_active_rate_aed,
       ROUND((SELECT median_active_rate_aed FROM median_price),2) AS median_active_rate_aed,
       ROUND(100.0*AVG(CASE WHEN f.active=1 THEN f.occupancy END),2) AS mean_provider_est_occupancy_pct,
       SUM(CASE WHEN f.active=1 AND f.occupancy=0 THEN 1 ELSE 0 END) AS active_zero_est_occupancy,
       ROUND(SUM(CASE WHEN f.active=1 THEN f.native_revenue ELSE 0 END),2) AS sum_provider_est_revenue_aed
FROM fact_listing_month f
WHERE f.month_date = ?
GROUP BY f.month_date
""", (latest_month, latest_month))
kpi = latest_kpis.iloc[0]
assert int(kpi.observed_listings) >= int(kpi.active_listings) >= 0
assert 0 <= float(kpi.mean_provider_est_occupancy_pct) <= 100
assert int(kpi.active_zero_est_occupancy) <= int(kpi.active_listings)

# Independent pandas median check against the original fact table.
active_rates_check = sql_frame("""
SELECT native_avg_daily_rate FROM fact_listing_month
WHERE month_date=? AND active=1 AND native_avg_daily_rate IS NOT NULL
""", (latest_month,))
assert np.isclose(
    float(kpi.median_active_rate_aed), active_rates_check.native_avg_daily_rate.median(), atol=.011
)
display(latest_kpis)
print(f"Latest active listings with zero provider-estimated occupancy: "
      f"{100*int(kpi.active_zero_est_occupancy)/int(kpi.active_listings):.1f}%")

## 4. Historical coverage and rate patterns

Changes in the observed listing population can create apparent market changes that reflect **sample composition**, not demand. The query uses `LAG()` to measure month-to-month coverage change and reconciles it with the precomputed coverage table. The median rate shown is the within-month median of **active, represented listings**, not a Dubai-wide price index.

In [4]:
history = sql_frame("""
WITH monthly AS (
 SELECT month_date, observed_listings, active_listings,
        median_active_adr_aed, median_active_estimated_occupancy,
        LAG(observed_listings) OVER (ORDER BY month_date) AS previous_observed
 FROM fact_coverage_month
)
SELECT month_date, observed_listings, active_listings,
       ROUND(median_active_adr_aed,2) AS median_active_adr_aed,
       ROUND(100.0*median_active_estimated_occupancy,2) AS median_provider_est_occupancy_pct,
       ROUND(100.0*(observed_listings*1.0 / NULLIF(previous_observed,0)-1),2)
           AS independent_coverage_change_pct
FROM monthly ORDER BY month_date
""")
coverage_view = sql_frame("""
 SELECT month_date, coverage_change_pct
 FROM vw_monthly_coverage_and_performance ORDER BY month_date
""")
coverage_join = history.merge(coverage_view, on="month_date", validate="one_to_one")
coverage_delta = (coverage_join.independent_coverage_change_pct -
                  coverage_join.coverage_change_pct).abs()
assert coverage_delta.dropna().max() < .011
assert len(history) == int(record.reporting_months)
assert (history.active_listings <= history.observed_listings).all()
large_changes = history.loc[history.independent_coverage_change_pct.abs() > 25].copy()
print("Reporting months:", len(history))
print("Months with absolute coverage change >25%:", len(large_changes))
display(large_changes[["month_date","observed_listings","independent_coverage_change_pct"]])

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(pd.to_datetime(history.month_date), history.median_active_adr_aed, linewidth=2)
ax.set(title="Median nightly rate of active listings in the observed sample",
       xlabel="Reporting month", ylabel="Median reported rate (AED)")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / "06_historical_median_rate.png", dpi=170, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(pd.to_datetime(history.month_date), history.observed_listings, linewidth=2)
ax.set(title="Changing sample coverage across reporting months",
       xlabel="Reporting month", ylabel="Observed listings (count)")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / "06_historical_sample_coverage.png", dpi=170, bbox_inches="tight")
plt.show()

## 5. Like-for-like monthly pricing: SQL window functions

A CTE with `LAG()` identifies each listing's previous record; a comparison is retained only when the previous observation is **the immediately preceding calendar month** and both months were active. This avoids treating gaps in coverage as consecutive comparisons. `ROW_NUMBER()` and `COUNT() OVER (PARTITION BY month_date)` produce the monthly **median**, which is less sensitive to extreme percentage movements than the mean.

The resulting cohort changes from month to month and cannot by itself prove that the whole Dubai market experienced the same rate changes.

In [5]:
pairwise_sql = """
WITH ordered AS (
  SELECT listing_id,month_date,active,native_avg_daily_rate,
         LAG(month_date) OVER (PARTITION BY listing_id ORDER BY month_date) AS previous_month,
         LAG(active) OVER (PARTITION BY listing_id ORDER BY month_date) AS previous_active,
         LAG(native_avg_daily_rate) OVER (PARTITION BY listing_id ORDER BY month_date) AS previous_rate
  FROM fact_listing_month
),
pairs AS (
  SELECT listing_id,month_date,
         100.0*(native_avg_daily_rate*1.0/previous_rate - 1) AS change_pct
  FROM ordered
  WHERE active=1 AND previous_active=1
    AND previous_month = DATE(month_date, '-1 month')
    AND native_avg_daily_rate IS NOT NULL AND previous_rate > 0
),
ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY month_date ORDER BY change_pct) AS rn,
            COUNT(*) OVER (PARTITION BY month_date) AS n
  FROM pairs
)
SELECT month_date, COUNT(*) AS matched_active_pairs,
       ROUND(AVG(change_pct),2) AS mean_rate_change_pct,
       ROUND(AVG(CASE WHEN rn IN ((n+1)/2,(n+2)/2) THEN change_pct END),2)
          AS median_rate_change_pct,
       SUM(CASE WHEN ABS(change_pct)>100 THEN 1 ELSE 0 END) AS extreme_over_100pct
FROM ranked
GROUP BY month_date ORDER BY month_date
"""
matched_monthly = sql_frame(pairwise_sql)
view_pairs = sql_frame("""
SELECT month_date,COUNT(*) AS view_pair_count
FROM vw_pairwise_same_listing_month
WHERE mom_rate_change_pct IS NOT NULL
GROUP BY month_date ORDER BY month_date
""")
matched_check = matched_monthly.merge(view_pairs, on="month_date", how="outer", validate="one_to_one")
assert matched_check[['matched_active_pairs','view_pair_count']].notna().all().all()
assert (matched_check.matched_active_pairs == matched_check.view_pair_count).all()
print('Consecutive active listing-month pairs:', int(matched_monthly.matched_active_pairs.sum()))
display(matched_monthly.tail(10))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(pd.to_datetime(matched_monthly.month_date), matched_monthly.mean_rate_change_pct,
        label='Mean', linewidth=1.8)
ax.plot(pd.to_datetime(matched_monthly.month_date), matched_monthly.median_rate_change_pct,
        label='Median', linewidth=2)
ax.axhline(0, linewidth=1, linestyle='--')
ax.set(title='Same-listing month-over-month rate change: mean versus median',
       xlabel='Reporting month', ylabel='Rate change (%)')
ax.legend(); ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / '06_matched_listing_mean_vs_median.png', dpi=170, bbox_inches='tight')
plt.show()

## 6. Extreme monthly movements and robustness

Percentage changes become unstable when the previous nightly rate is very small. This section measures their frequency and compares the **mean**, **median**, and a **5% tail-trimmed mean** across the matched listing pairs. Tail trimming describes robustness; it is **not** an instruction to delete legitimate high-priced listings. No source data are modified.

In [6]:
pair_distribution = sql_frame("""
WITH ordered AS (
 SELECT listing_id,month_date,active,native_avg_daily_rate,
        LAG(month_date) OVER(PARTITION BY listing_id ORDER BY month_date) AS prev_month,
        LAG(active) OVER(PARTITION BY listing_id ORDER BY month_date) AS prev_active,
        LAG(native_avg_daily_rate) OVER(PARTITION BY listing_id ORDER BY month_date) AS prev_rate
 FROM fact_listing_month
), pairs AS (
 SELECT 100.0*(native_avg_daily_rate*1.0/prev_rate-1) AS change_pct
 FROM ordered WHERE active=1 AND prev_active=1
   AND prev_month=DATE(month_date,'-1 month') AND prev_rate>0
   AND native_avg_daily_rate IS NOT NULL
), ranked AS (
 SELECT change_pct,ROW_NUMBER() OVER (ORDER BY change_pct) AS rn,COUNT(*) OVER() AS n
 FROM pairs
)
SELECT COUNT(*) AS pair_count,
       ROUND(AVG(change_pct),2) AS mean_change_pct,
       ROUND(AVG(CASE WHEN rn IN ((n+1)/2,(n+2)/2) THEN change_pct END),2)
         AS median_change_pct,
       ROUND(AVG(CASE WHEN rn > CAST(n*0.05 AS INTEGER)
                      AND rn <= n-CAST(n*0.05 AS INTEGER) THEN change_pct END),2)
         AS trimmed_mean_5pct_each_tail,
       ROUND(MIN(change_pct),2) AS minimum_change_pct,
       ROUND(MAX(change_pct),2) AS maximum_change_pct,
       SUM(CASE WHEN change_pct>100 THEN 1 ELSE 0 END) AS increases_over_100pct,
       SUM(CASE WHEN change_pct < -50 THEN 1 ELSE 0 END) AS decreases_below_minus_50pct
FROM ranked
""")
assert int(pair_distribution.pair_count.iloc[0]) == int(matched_monthly.matched_active_pairs.sum())
display(pair_distribution)
print('Distribution uses unrounded rate changes; Notebook 05 view rounds individual values to 2 decimals.')

## 7. Property segments: CTEs, ranking, and denominator controls

The latest reporting month is segmented using **room type** and **bedroom count**. SQL `ROW_NUMBER() OVER (PARTITION BY room_type)` ranks bedroom groups within each accommodation category by representation. Small cohorts are reported but not treated as robust peer benchmarks. The grouped mean is descriptive; detailed matching remains in Notebook 04.

In [7]:
segments = sql_frame("""
WITH latest_active AS (
 SELECT listing_id,room_type,bedrooms,native_avg_daily_rate,occupancy
 FROM fact_listing_month
 WHERE month_date=? AND active=1 AND bedrooms IS NOT NULL
), segments AS (
 SELECT room_type,bedrooms,COUNT(*) AS active_listings,
        ROUND(AVG(native_avg_daily_rate),2) AS mean_rate_aed,
        ROUND(100.0*AVG(occupancy),2) AS mean_provider_est_occupancy_pct
 FROM latest_active GROUP BY room_type,bedrooms
), ranked AS (
 SELECT *,ROW_NUMBER() OVER (PARTITION BY room_type
                           ORDER BY active_listings DESC,bedrooms) AS rank_within_room_type
 FROM segments
)
SELECT * FROM ranked ORDER BY active_listings DESC,room_type,bedrooms LIMIT 15
""", (latest_month,))
assert (segments.active_listings > 0).all()
assert (segments.rank_within_room_type >= 1).all()
display(segments)

# Segment reconciliation is conducted against every group, not just the top 15 displayed.
segment_count = int(sql_frame("""
SELECT COUNT(*) AS n FROM fact_listing_month
WHERE month_date=? AND active=1 AND bedrooms IS NOT NULL
""", (latest_month,)).iloc[0,0])
view_segment_count = int(sql_frame("""
SELECT SUM(active_sample_listings) AS n FROM vw_segment_monthly_rates
WHERE month_date=?
""", (latest_month,)).iloc[0,0])
assert segment_count == view_segment_count
print('Active listings with bedroom metadata, independently reconciled:', segment_count)

## 8. Competitor pricing review: directional alerts

The peer-matching engine (Notebook 04) screens listings against comparable properties. This query uses `CASE` and conditional aggregates to separate **higher-than-peer**, **lower-than-peer**, and **unassessed** observations. The 20% review threshold is an analyst-defined screening rule, not a validated revenue-optimizing threshold. All totals are reconciled against the independent benchmark fact table.

In [8]:
benchmark_status = sql_frame("""
WITH latest AS (
 SELECT comparison_status,review_signal,premium_pct,confidence_label,peer_count
 FROM fact_peer_benchmark WHERE month_date=?
), grouped AS (
 SELECT CASE
    WHEN comparison_status!='Eligible' THEN 'Unassessed'
    WHEN review_signal NOT IN ('Review','Large deviation') THEN 'Within threshold'
    WHEN premium_pct > 0 THEN 'Above peer benchmark'
    WHEN premium_pct < 0 THEN 'Below peer benchmark'
    ELSE 'Requires further classification'
   END AS pricing_assessment,
   COUNT(*) AS listings,
   ROUND(AVG(CASE WHEN comparison_status='Eligible' THEN peer_count END),1) AS avg_peer_count
 FROM latest
 GROUP BY 1
)
SELECT * FROM grouped ORDER BY CASE pricing_assessment
 WHEN 'Above peer benchmark' THEN 1 WHEN 'Below peer benchmark' THEN 2
 WHEN 'Within threshold' THEN 3 WHEN 'Unassessed' THEN 4 ELSE 5 END
""", (latest_month,))
queue_count = int(sql_frame('SELECT COUNT(*) AS n FROM vw_latest_price_review_queue').iloc[0,0])
expected_queue = int(sql_frame("""
SELECT COUNT(*) AS n FROM fact_peer_benchmark
WHERE month_date=? AND comparison_status='Eligible'
  AND review_signal IN ('Review','Large deviation')
""",(latest_month,)).iloc[0,0])
assert queue_count == expected_queue
assert int(benchmark_status.listings.sum()) == int(kpi.observed_listings)
above = int(benchmark_status.loc[benchmark_status.pricing_assessment=='Above peer benchmark','listings'].sum())
below = int(benchmark_status.loc[benchmark_status.pricing_assessment=='Below peer benchmark','listings'].sum())
assert above + below == queue_count
assert int(benchmark_status.loc[benchmark_status.pricing_assessment=='Unassessed','listings'].sum()) >= 0
print('Review candidates:', queue_count)
display(benchmark_status)

fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.bar(benchmark_status.pricing_assessment, benchmark_status.listings)
ax.set(title='Latest-month competitor pricing assessment (observed listings)',
       xlabel='Analytical screening category', ylabel='Properties (count)')
ax.tick_params(axis='x', rotation=14)
ax.grid(axis='y', alpha=.15)
fig.tight_layout()
fig.savefig(FIGURES / '06_competitor_review_directions.png', dpi=170, bbox_inches='tight')
plt.show()

## 9. Independent business-logic validation

The checks below reconcile the SQL outputs against original fact tables, rather than merely checking whether views execute. They also verify view membership, denominator logic, rate-change calculations, and the latest-period filter. All validations must pass before exporting results.

In [9]:
reconciliation = sql_frame("""
WITH queue AS (
 SELECT listing_id,month_date FROM vw_latest_price_review_queue
), expected AS (
 SELECT listing_id,month_date FROM fact_peer_benchmark
 WHERE month_date = (SELECT MAX(month_date) FROM fact_listing_month)
   AND comparison_status='Eligible' AND review_signal IN ('Review','Large deviation')
), discrepancies AS (
 SELECT listing_id,month_date FROM queue EXCEPT SELECT listing_id,month_date FROM expected
), missing AS (
 SELECT listing_id,month_date FROM expected EXCEPT SELECT listing_id,month_date FROM queue
)
SELECT (SELECT COUNT(*) FROM discrepancies) AS extra_queue_keys,
       (SELECT COUNT(*) FROM missing) AS missing_queue_keys,
       (SELECT COUNT(*) FROM vw_latest_price_review_queue) AS actual_queue_rows,
       (SELECT COUNT(*) FROM expected) AS expected_queue_rows
""")
assert int(reconciliation.extra_queue_keys.iloc[0]) == 0
assert int(reconciliation.missing_queue_keys.iloc[0]) == 0

# Recalculate the underlying month-on-month change on a representative,
# deterministic set of matched pairs. Comparisons to the view allow 0.01 pp rounding.
pair_recheck = sql_frame("""
WITH matched AS (
 SELECT listing_id,month_date,native_avg_daily_rate,prev_rate,mom_rate_change_pct
 FROM vw_pairwise_same_listing_month
 WHERE mom_rate_change_pct IS NOT NULL
 ORDER BY listing_id,month_date LIMIT 500
)
SELECT COUNT(*) AS checked_rows,
 SUM(CASE WHEN ABS(ROUND(100.0*(native_avg_daily_rate*1.0/prev_rate-1),2)
           -mom_rate_change_pct) > 0.011 THEN 1 ELSE 0 END) AS rate_change_mismatches
FROM matched
""")
assert int(pair_recheck.checked_rows.iloc[0]) == min(500, int(matched_monthly.matched_active_pairs.sum()))
assert int(pair_recheck.rate_change_mismatches.iloc[0]) == 0

test_results = pd.DataFrame([
    ('SQLite physical integrity', True),
    ('Foreign-key consistency', True),
    ('Fact and benchmark row reconciliation', int(record.observations)==int(record.benchmark_rows)),
    ('Monthly coverage independently reconciled', int(coverage_key_issues)==0),
    ('Active <= observed each month', bool((history.active_listings<=history.observed_listings).all())),
    ('Coverage LAG recomputation matches view', bool(coverage_delta.dropna().max()<.011)),
    ('Consecutive-month peer counts match SQL view', bool((matched_check.matched_active_pairs == matched_check.view_pair_count).all())),
    ('Segment totals independently reconciled', segment_count==view_segment_count),
    ('Latest review queue exact key match', int(reconciliation.extra_queue_keys.iloc[0])==0 and int(reconciliation.missing_queue_keys.iloc[0])==0),
    ('Independent price-change arithmetic', int(pair_recheck.rate_change_mismatches.iloc[0])==0),
],columns=['validation','passed'])
assert test_results.passed.all(), 'One or more business checks failed.'
display(test_results)
print(f"Validation: {len(test_results)}/{len(test_results)} passed")

## 10. Reproducible exports and business interpretation

Aggregated outputs below can be used in the GitHub repository and in Power BI documentation, subject to the source license. They are **not** a substitute for the data model produced by Notebook 05. Export files retain explicit sample-focused terminology.

In [10]:
exports = {
    '06_latest_sample_kpis.csv': latest_kpis,
    '06_monthly_sample_history.csv': history,
    '06_matched_listing_rate_changes.csv': matched_monthly,
    '06_rate_change_robustness.csv': pair_distribution,
    '06_latest_segments.csv': segments,
    '06_latest_competitor_screening.csv': benchmark_status,
    '06_validation_results.csv': test_results,
}
for filename, frame in exports.items():
    frame.to_csv(REPORT_SQL / filename, index=False)

print(f"Reporting period: {record.first_month} through {latest_month}")
print(f"Observations: {int(record.observations):,}; unique listings: {int(record.unique_listings):,}")
print(f"Latest month: {int(kpi.observed_listings):,} observed, {int(kpi.active_listings):,} active listings")
print(f"Mean/median reported active ADR: AED {kpi.mean_active_rate_aed:,.2f} / AED {kpi.median_active_rate_aed:,.2f}")
print(f"Provider-estimated mean occupancy: {kpi.mean_provider_est_occupancy_pct:.2f}%")
print(f"Model-derived revenue sum for active sampled listings: AED {kpi.sum_provider_est_revenue_aed:,.0f}")
print(f"Coverage shifts >25%: {len(large_changes)} months")
print(f"Matched consecutive-month listing pairs: {int(pair_distribution.pair_count.iloc[0]):,}")
print(f"Pair rate change mean / median / trimmed mean (5% each tail): "
      f"{pair_distribution.mean_change_pct.iloc[0]:.2f}% / "
      f"{pair_distribution.median_change_pct.iloc[0]:.2f}% / "
      f"{pair_distribution.trimmed_mean_5pct_each_tail.iloc[0]:.2f}%")
print(f"Competitor review candidates: {queue_count:,} ({above:,} above; {below:,} below peer benchmark)")
print(f"Saved {len(exports)} CSV summaries and 4 figures under reports/")
conn.close()

## Key findings and analytical decisions

**Observed sample, not market totals.** The dataset spans 60 monthly periods and contains 99,745 listing-month records for 4,364 distinct listing IDs. The latest period is **February 2026**. The saved SQL outputs above provide the precise validated counts and measures for this release.

**Pricing is skewed.** The latest sample's mean active nightly rate exceeds its median. In addition, the 71,960 valid consecutive active listing-month comparisons contain pronounced outliers. Therefore, median and robustness-aware statistics should accompany means in any Power BI historical comparison.

**Sample coverage changes materially.** Five reporting periods experienced an absolute change in observed listings greater than 25%. A rise or fall in aggregate estimated revenue cannot be treated as Dubai-wide growth without a coverage-aware methodology.

**Pricing-review signals need human interpretation.** In February 2026, 902 listings are flagged by analyst-defined competitor thresholds: 471 above and 431 below their peer median. These are **screening candidates**, not proven overpricing/underpricing; changes in accommodation quality, guest capacity and amenities may explain the differences.

**SQL design and next steps.** This notebook independently reconciles source-fact records, monthly coverage, same-listing rate changes, SQL analytical views and review-queue membership. Its aggregate outputs support the Executive Command Center, Market Trends and Competitive Pricing Studio. Notebook 07 will validate Power BI exports and check measures and relationships before dashboard development.

**Limitations:** Listing representation varies by month; neighborhood coverage is incomplete; AirROI occupancy and revenue are modeled estimates; peer-matching does not fully control for amenities, quality or micro-location. No causal pricing or profitability claims are warranted.

## Standalone SQL extension: fixed cohorts, dispersion and action priorities

The SQL directory is independently runnable. A fixed 12-month cohort controls membership but introduces survivor bias. Segment quartiles use nearest-rank; peer quartiles retain Python's linear interpolation. The review-priority view places sufficiently covered, narrower-dispersion deviations first, sends broader/heterogeneous matches to comparability review, and retains missing-coverage cases. These are operational heuristics, not validated predictions.

In [11]:
import runpy
advanced_audit=runpy.run_path(str(PROJECT_ROOT/'src/05_run_sql_analysis.py'))['run']()
for filename in ('04b_fixed_cohort','05_property_segmentation','07_opportunity_analysis'):
    print(filename)
    display(pd.read_csv(REPORT_SQL/(filename+'.csv')))
print('Independent reconciliation:', {k:advanced_audit[k] for k in ('adjacent_active_pairs','segment_groups_reconciled','fixed_12month_cohort','priority_rows_reconciled')})


In [12]:
from IPython.display import Markdown
cohort=pd.read_csv(REPORT_SQL/'04b_fixed_cohort.csv')
first,last=cohort.iloc[0],cohort.iloc[-1]
display(Markdown(f"The stable cohort retains **{int(last.cohort_listings):,}** properties. "
    f"Its median ADR moves from **AED {first.cohort_median_aed:,.0f}** to **AED {last.cohort_median_aed:,.0f}**, "
    f"while the changing active sample moves from **AED {first.full_sample_median_aed:,.0f}** "
    f"to **AED {last.full_sample_median_aed:,.0f}**. "
    "These endpoints cover different calendar months, so seasonality and survivor selection remain relevant. "
    "Managers should inspect the action workload above alongside peer dispersion and source quality before changing prices."))
